<a href="https://colab.research.google.com/github/dashy0070/Cybersecurity_AI/blob/main/NIDS_E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NIDS

Prompt: Load the KDD Cup 1999 dataset into a pandas DataFrame.
Display:
1. Dataset shape
2. Column names
3. First 5 rows
4. Data types
5. Distribution of the target label

import kagglehub

# Download latest version
path = kagglehub.dataset_download("galaxyh/kdd-cup-1999-data")

print("Path to dataset files:", path)


In [1]:
import pandas as pd
import os
import kagglehub

# Download latest version of the KDD Cup 1999 dataset and declare 'path' here
path = kagglehub.dataset_download("galaxyh/kdd-cup-1999-data")
print("Path to dataset files:", path)

# Define column names for the KDD Cup 1999 dataset
# Based on the kddcup.names file, there are 41 features and 1 label.
column_names = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
    'logged_in', 'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells', 'num_access_files',
    'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate',
    'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label'
]

# Construct the full path to the KDD Cup 1999 10% data file.
# The dataset typically contains 'kddcup.data_10_percent.gz'.
file_path = os.path.join(path, 'kddcup.data_10_percent.gz')

print(f"Loading dataset from: {file_path}")

try:
    # Load the gzipped CSV into a pandas DataFrame
    df = pd.read_csv(file_path, compression='gzip', header=None, names=column_names)
    print("Dataset loaded successfully!")

    # 1. Dataset shape
    print("\n1. Dataset Shape:")
    print(df.shape)

    # 2. Column names
    print("\n2. Column Names:")
    print(df.columns.tolist())

    # 3. First 5 rows
    print("\n3. First 5 Rows:")
    print(df.head())

    # 4. Data types
    print("\n4. Data Types:")
    df.info()

    # 5. Distribution of the target label
    print("\n5. Distribution of the Target Label ('label' column):")
    print(df['label'].value_counts())

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the dataset was downloaded correctly and the file exists.")
except Exception as e:
    print(f"An unexpected error occurred while loading or processing the dataset: {e}")


100%|██████████| 87.8M/87.8M [00:05<00:00, 17.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/galaxyh/kdd-cup-1999-data/versions/1
Loading dataset from: /root/.cache/kagglehub/datasets/galaxyh/kdd-cup-1999-data/versions/1/kddcup.data_10_percent.gz
Dataset loaded successfully!

1. Dataset Shape:
(494021, 42)

2. Column Names:
['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_s

Inference:

### Data Preprocessing

Prompt: Perform preprocessing on the KDD Cup 1999 dataset:

1. Convert the label column into binary classes (0 for normal, 1 for attack).
2. Encode categorical columns such as protocol_type, service, and flag using LabelEncoder or OneHotEncoding.
3. Handle missing values if present.
4. Separate the dataset into X (features) and y (target).
5. Display the shape of X and y.

In [2]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

# Make a copy of the DataFrame to avoid modifying the original 'df' in place
df_processed = df.copy()

# 1. Convert the label column into binary classes (0 for normal, 1 for attack).
# 'normal.' is considered normal (0), all other labels are attacks (1).
# Remove the trailing dot from labels for consistency if present.
df_processed['label'] = df_processed['label'].str.replace('.', '', regex=False)
y = df_processed['label'].apply(lambda x: 0 if x == 'normal' else 1)

# 4. Separate the dataset into X (features) and y (target)
# X will be the features, excluding the original 'label' column
X = df_processed.drop('label', axis=1)

# Identify categorical columns for encoding
categorical_features = ['protocol_type', 'service', 'flag']

# Create a ColumnTransformer to apply OneHotEncoder to categorical features
# and passthrough the remaining numerical features.
# sparse_output=False ensures a dense numpy array output.
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ],
    remainder='passthrough' # Keep all other columns (numerical) as they are
)

# Apply the ColumnTransformer to X
X_encoded = preprocessor.fit_transform(X)

# Get the feature names after one-hot encoding
onehot_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)

# Get the names of the columns that were passed through (numerical features)
# Need to filter out the categorical_features from the original X columns
remaining_feature_names = X.columns.drop(categorical_features).tolist()

# Combine all feature names
all_feature_names = list(onehot_feature_names) + remaining_feature_names

# Convert the numpy array X_encoded back to a DataFrame with proper column names
X = pd.DataFrame(X_encoded, columns=all_feature_names, index=X.index)

# 3. Handle missing values if present.
# Based on the df.info() from previous execution, there are no missing values.
# We can confirm this again, though it's not strictly necessary to add code if none are found.
if X.isnull().sum().sum() > 0:
    print("Warning: Missing values found in X after encoding. Further handling might be required.")
else:
    print("No missing values found in the feature set X.")

# 5. Display the shape of X and y
print(f"\nShape of X (features): {X.shape}")
print(f"Shape of y (target): {y.shape}")

print("\nFirst 5 rows of X (features) after preprocessing:")
print(X.head())

print("\nFirst 5 rows of y (target):")
print(y.head())


No missing values found in the feature set X.

Shape of X (features): (494021, 118)
Shape of y (target): (494021,)

First 5 rows of X (features) after preprocessing:
   protocol_type_icmp  protocol_type_tcp  protocol_type_udp  service_IRC  \
0                 0.0                1.0                0.0          0.0   
1                 0.0                1.0                0.0          0.0   
2                 0.0                1.0                0.0          0.0   
3                 0.0                1.0                0.0          0.0   
4                 0.0                1.0                0.0          0.0   

   service_X11  service_Z39_50  service_auth  service_bgp  service_courier  \
0          0.0             0.0           0.0          0.0              0.0   
1          0.0             0.0           0.0          0.0              0.0   
2          0.0             0.0           0.0          0.0              0.0   
3          0.0             0.0           0.0          0.0        

### Feature Scaling

Prompt: Apply StandardScaler to normalize all numerical features in X.
Do not scale the target variable y.
Display the first five rows of the scaled features.

In [3]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Initialize StandardScaler
scaler = StandardScaler()

# Apply StandardScaler to X
# StandardScaler returns a numpy array, so we convert it back to a DataFrame
X_scaled_array = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns, index=X.index)

print("Shape of X_scaled (features after scaling):", X_scaled.shape)

# Display the first five rows of the scaled features
print("\nFirst 5 rows of scaled X (features):")
print(X_scaled.head())


Shape of X_scaled (features after scaling): (494021, 118)

First 5 rows of scaled X (features):
   protocol_type_icmp  protocol_type_tcp  protocol_type_udp  service_IRC  \
0           -1.160946           1.264603          -0.207295     -0.00933   
1           -1.160946           1.264603          -0.207295     -0.00933   
2           -1.160946           1.264603          -0.207295     -0.00933   
3           -1.160946           1.264603          -0.207295     -0.00933   
4           -1.160946           1.264603          -0.207295     -0.00933   

   service_X11  service_Z39_50  service_auth  service_bgp  service_courier  \
0    -0.004719       -0.013648     -0.025776     -0.01465        -0.014787   
1    -0.004719       -0.013648     -0.025776     -0.01465        -0.014787   
2    -0.004719       -0.013648     -0.025776     -0.01465        -0.014787   
3    -0.004719       -0.013648     -0.025776     -0.01465        -0.014787   
4    -0.004719       -0.013648     -0.025776     -0.01465

Inference:

### Train-Test Split

Prompt: Split the dataset into training and testing sets using an 80/20 ratio.

Use stratified splitting based on the target variable to maintain class distribution.
Set random_state=42 for reproducibility.

Display the shapes of:
X_train,
X_test,
y_train,
y_test,

In [4]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets (80/20 ratio)
# Use stratified splitting based on 'y' to maintain class distribution
# Set random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)

# Display the shapes of the resulting sets
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (395216, 118)
Shape of X_test: (98805, 118)
Shape of y_train: (395216,)
Shape of y_test: (98805,)


### Train the model

Prompt: Train traditional supervised machine learning models for intrusion detection using the training dataset.

Implement the following models:
1. Logistic Regression
2. Naive Bayes
3. Decision Tree

Train each model and store the predictions on the test dataset.

### LR Model

In [5]:
from sklearn.linear_model import LogisticRegression
import time

# Initialize dictionaries to store trained models and their predictions
# if they don't already exist from a previous execution in the current session.
if 'models' not in locals():
    models = {}
if 'predictions' not in locals():
    predictions = {}

# 1. Logistic Regression
print("\nTraining Logistic Regression model...")
start_time = time.time()
log_reg_model = LogisticRegression(random_state=42, solver='liblinear', n_jobs=-1) # n_jobs=-1 to use all available cores
log_reg_model.fit(X_train, y_train)
log_reg_train_time = time.time() - start_time
models['Logistic Regression'] = log_reg_model
print(f"Logistic Regression trained in {log_reg_train_time:.2f} seconds.")

print("Making predictions with Logistic Regression...")
log_reg_predictions = log_reg_model.predict(X_test)
predictions['Logistic Regression'] = log_reg_predictions
print("Logistic Regression predictions stored.")

print("\nLogistic Regression training and prediction complete.")



Training Logistic Regression model...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 2.
  warnings.warn(


Logistic Regression trained in 89.14 seconds.
Making predictions with Logistic Regression...
Logistic Regression predictions stored.

Logistic Regression training and prediction complete.


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


### Naive Bayes Model

In [6]:
from sklearn.naive_bayes import GaussianNB

# 2. Naive Bayes
print("\nTraining Naive Bayes model...")
start_time = time.time()
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_train_time = time.time() - start_time
models['Naive Bayes'] = nb_model
print(f"Naive Bayes trained in {nb_train_time:.2f} seconds.")

print("Making predictions with Naive Bayes...")
nb_predictions = nb_model.predict(X_test)
predictions['Naive Bayes'] = nb_predictions
print("Naive Bayes predictions stored.")

print("\nNaive Bayes training and prediction complete.")


Training Naive Bayes model...
Naive Bayes trained in 0.81 seconds.
Making predictions with Naive Bayes...
Naive Bayes predictions stored.

Naive Bayes training and prediction complete.


### DT Model

In [7]:
from sklearn.tree import DecisionTreeClassifier

# 3. Decision Tree
print("\nTraining Decision Tree model...")
start_time = time.time()
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_train_time = time.time() - start_time
models['Decision Tree'] = dt_model
print(f"Decision Tree trained in {dt_train_time:.2f} seconds.")

print("Making predictions with Decision Tree...")
dt_predictions = dt_model.predict(X_test)
predictions['Decision Tree'] = dt_predictions
print("Decision Tree predictions stored.")

print("\nDecision Tree training and prediction complete.")


Training Decision Tree model...
Decision Tree trained in 4.88 seconds.
Making predictions with Decision Tree...
Decision Tree predictions stored.

Decision Tree training and prediction complete.


Inference:

### Evaluate the model

Evaluate the performance of DT, LR and Naive bayes model

Generate the following metrics for each model:
1. Accuracy
2. Precision
3. Recall
4. F1-score
5. Confusion matrix

Display the results in a comparison table.

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd
import numpy as np

# Initialize lists to store metrics for comparison
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []
confusion_matrices = {}

print("\nEvaluating Model Performance:")
for model_name, preds in predictions.items():
    print(f"\n--- {model_name} ---")

    # Calculate metrics
    accuracy = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    cm = confusion_matrix(y_test, preds)

    # Store metrics
    accuracy_scores.append({'Model': model_name, 'Metric': 'Accuracy', 'Score': accuracy})
    precision_scores.append({'Model': model_name, 'Metric': 'Precision', 'Score': precision})
    recall_scores.append({'Model': model_name, 'Metric': 'Recall', 'Score': recall})
    f1_scores.append({'Model': model_name, 'Metric': 'F1-Score', 'Score': f1})
    confusion_matrices[model_name] = cm

    # Print individual metrics
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)

# Create a comparison table for the main metrics
comparison_df = pd.DataFrame(accuracy_scores + precision_scores + recall_scores + f1_scores)
comparison_table = comparison_df.pivot(index='Model', columns='Metric', values='Score')

print("\n--- Model Performance Comparison Table ---")
print(comparison_table.to_markdown(numalign="left", stralign="left"))



Evaluating Model Performance:

--- Logistic Regression ---
Accuracy: 0.9985
Precision: 0.9995
Recall: 0.9987
F1-Score: 0.9991
Confusion Matrix:
[[19417    39]
 [  105 79244]]

--- Naive Bayes ---
Accuracy: 0.9476
Precision: 0.9997
Recall: 0.9351
F1-Score: 0.9663
Confusion Matrix:
[[19435    21]
 [ 5153 74196]]

--- Decision Tree ---
Accuracy: 0.9998
Precision: 0.9999
Recall: 0.9999
F1-Score: 0.9999
Confusion Matrix:
[[19445    11]
 [   11 79338]]

--- Model Performance Comparison Table ---
| Model               | Accuracy   | F1-Score   | Precision   | Recall   |
|:--------------------|:-----------|:-----------|:------------|:---------|
| Decision Tree       | 0.999777   | 0.999861   | 0.999861    | 0.999861 |
| Logistic Regression | 0.998543   | 0.999092   | 0.999508    | 0.998677 |
| Naive Bayes         | 0.947634   | 0.966308   | 0.999717    | 0.935059 |


### False Positive Rate (FPR) Analysis

In [9]:
# Calculate False Positive Rate (FPR) for each model
fpr_scores = {}
print("\n--- False Positive Rates (FPR) ---")
for model_name, cm in confusion_matrices.items():
    tn, fp, fn, tp = cm.ravel() # Unpack confusion matrix values

    # FPR = FP / (FP + TN)
    # Avoid division by zero if FP + TN is 0 (though unlikely in this context)
    if (fp + tn) > 0:
        fpr = fp / (fp + tn)
    else:
        fpr = 0.0 # Or np.nan, depending on desired handling of this edge case

    fpr_scores[model_name] = fpr
    print(f"{model_name}: {fpr:.6f}")

# Identify model with lowest FPR
lowest_fpr_model = min(fpr_scores, key=fpr_scores.get)
print(f"\nModel with the lowest FPR: {lowest_fpr_model} ({fpr_scores[lowest_fpr_model]:.6f})")

# Provide detailed analysis



--- False Positive Rates (FPR) ---
Logistic Regression: 0.002005
Naive Bayes: 0.001079
Decision Tree: 0.000565

Model with the lowest FPR: Decision Tree (0.000565)


### Performance Review (Decision Tree vs. Logistic Regression)

From the evaluation results, we have the following metrics for the Decision Tree and Logistic Regression models:

**Decision Tree:**
- **Accuracy:** 0.9998
- **Precision:** 0.9999
- **Recall:** 0.9999
- **F1-Score:** 0.9999
- **False Positive Rate (FPR):** 0.000565

**Logistic Regression:**
- **Accuracy:** 0.9985
- **Precision:** 0.9995
- **Recall:** 0.9987
- **F1-Score:** 0.9991
- **False Positive Rate (FPR):** 0.002005

**Summary:**
The Decision Tree model significantly outperformed Logistic Regression across all metrics, achieving near-perfect scores for accuracy, precision, recall, and F1-score, and a remarkably low False Positive Rate. This suggests that for this specific dataset and preprocessing, a Decision Tree-based model is highly effective in identifying network intrusions with minimal false alarms, which is crucial for high-traffic networks.

### Comparative Analysis: Decision Trees vs. Neural Networks for IDS in High-Traffic Networks

Based on the observed performance and the inherent characteristics of Decision Tree-based models and Neural Networks, we can draw a comparative analysis for their application in Intrusion Detection Systems in high-traffic environments.

**Decision Tree-based Models:**

*   **Effectiveness:** As demonstrated by the empirical results, a well-tuned Decision Tree can achieve exceptionally high accuracy, precision, recall, F1-score, and very low False Positive Rates. This makes them highly effective for the classification of network traffic as normal or anomalous.
*   **Pros:**
    *   **Interpretability:** This is a major advantage in IDS. Security analysts can easily understand the rules the model uses to classify an event, aiding in investigation and policy refinement. The "white-box" nature is invaluable.
    *   **Efficiency:** Fast training and prediction times make them suitable for real-time detection in high-traffic networks.
    *   **Data Handling:** Can naturally handle mixed data types without extensive preprocessing, and are not sensitive to feature scaling.
*   **Cons:**
    *   **Overfitting:** Prone to overfitting if not properly pruned. This can lead to memorization of training data, resulting in poor generalization and potentially high false alarms or missed attacks in a dynamic network environment.
    *   **Instability:** Sensitive to small changes in data, which might necessitate retraining or careful monitoring in evolving threat landscapes.
*   **Overfitting Prevention:** Effective strategies include pre-pruning (e.g., `max_depth`, `min_samples_leaf`), post-pruning (`ccp_alpha`), and using ensemble methods like Random Forests or Gradient Boosting, which significantly mitigate the overfitting risk and improve robustness.

**Neural Networks:**

*   **Effectiveness (Theoretical):** While not explicitly trained in this notebook, NNs are theoretically capable of achieving very high effectiveness, especially in detecting sophisticated and novel attack patterns due to their ability to learn complex, abstract representations.
*   **Pros:**
    *   **Complex Pattern Recognition:** Excellent at identifying intricate, non-linear relationships and evolving attack signatures.
    *   **Feature Learning:** Can automatically extract relevant features, reducing the need for manual feature engineering, which is a significant advantage in rapidly changing network environments.
    *   **Adaptability:** Highly adaptable to different types of network data and can potentially learn from vast amounts of data.
*   **Cons:**
    *   **"Black-Box" Nature:** The lack of interpretability is a major drawback. Understanding *why* an NN flagged an event as an attack is difficult, impeding forensic analysis and trust in the system.
    *   **Computational Demands:** High training and inference costs, especially for deep models, can be a challenge for deployment in resource-constrained environments or for very high-speed, real-time IDS.
    *   **Data Hunger:** Require massive amounts of labeled data, which is often difficult and expensive to acquire and maintain for IDS.
    *   **Overfitting:** Highly susceptible to overfitting, particularly with complex architectures, making robust generalization challenging.
*   **Overfitting Prevention:** Techniques like L1/L2 regularization, dropout, early stopping, batch normalization, and data augmentation are crucial for NNs to generalize well and prevent them from merely memorizing training data.

**Conclusion for High-Traffic IDS:**

For high-traffic network Intrusion Detection Systems, both Decision Tree-based models and Neural Networks offer compelling advantages but also present challenges. The Decision Tree, as demonstrated, offers a highly effective and *interpretable* solution with excellent performance and manageable overfitting through pruning and ensemble methods. Its efficiency makes it practical for real-time scenarios.

Neural Networks, while potentially superior in detecting novel and complex threats due to their advanced pattern learning, face significant hurdles related to their "black-box" nature, high computational costs, and substantial data requirements. The lack of interpretability is particularly critical in security, where understanding attack vectors is paramount.

Therefore, while NNs might be suited for advanced, specialized threat detection with ample resources and data, Decision Tree-based models (especially in ensembles like Random Forests or Gradient Boosting) often strike a better balance between performance, interpretability, and operational feasibility for general-purpose IDS in high-traffic environments.